# Cell Cycle Classification

## Imports

In [1]:
%matplotlib inline

In [ ]:
from __future__ import print_function
import h5py
import sys
import os
from os.path import exists
sys.path.append('../')
sys.path.append('../core')
import scipy as SP
from cyclone import cyclone
from load_data import *
import joblib

CFG = {}
CFG['test_file'] = '../../data/data_mESCbulk.h5f'
#CFG['test_file'] = './data/normCounts_mESCquartz.h5f'
CFG['train_file'] = '../../data/normCountsBuettnerEtAl.h5f'
CFG['util_file'] = '../../data/normCounts_mESCquartz.h5f'

out_dir = './out_mESCbulk/'

## Data loading

In [ ]:
data = load_data(CFG, is_Ens=True, het_only=True, het_onlyCB=False, gene_set='GOCB')#gene_set can be either a list of genes,
class_labels = data['class_labels']#['G1','G2M','S']#['T-cells']#d#['Liver']#['early', 'late', 'mid']#data['class_labels']#['G1', 'S','G2M']#['Liver']#[data['class_labels']#['T-cells']##['G1', 'S','G2M']#['T-cells']#['G1', 'S','G2M']# #                                                                                        #or 'all' (all genes), 'GOCB' GO and cyclebase or 'CB' or 'GO'
data.keys()

print(data['cc_ens'].shape[0], 'Cell cycle genes used for training and prediction')
print(data['class_labels'])

## Model training

In [ ]:
cyclone = cyclone(data['Y'],row_namesY= data['genes'],cc_geneNames= data['cc_ens'],labels = data['labels'], 
                         Y_tst = data['Y_test'], row_namesY_tst = data['genes_tst'], labels_tst = data['labels_tst'])

In [ ]:
cyclone.trainModel(rftop = 40, cv=2, out_dir = out_dir, do_pca=1, npc=1, is_SVM=0)

### Model exporting/loading

In [ ]:
# after trainModel()
joblib.dump(cyclone, 'cyclone_model.joblib')

In [ ]:
# later
cyclone = joblib.load('cyclone_model.joblib')

In [ ]:
# reuse stored predictions
pred = cyclone.scores_tst.argmax(axis=1) + 1   # RF on test set

# replot
cyclone.plotHistograms(out_dir='./out', method='RF')
cyclo

In [ ]:
ne.plotScatter(out_dir='./out', method='RF')

## Training analysis

In [ ]:
cyclone.plotHistograms(class_labels = class_labels, out_dir = out_dir, method='GNB', do_h=True)
cyclone.plotPerformance(plot_test=False, out_dir =out_dir, method='GNB')

In [ ]:
cyclone.plotScatter(plot_test = True, xaxis = 0, yaxis = 2, xlab = 'G1 score', ylab = 'G2M score', class_labels = class_labels, out_dir = out_dir, method='GNB')
cyclone.plotScatter(plot_test = False, xaxis = 0, yaxis = 2, xlab = 'G1 score', ylab = 'G2M score', class_labels = ['G1', 'S', 'G2M'], out_dir = out_dir, method='GNB')

## Wyniki cyklu komórkowego dla realnych komórek HSPC (`scran::cyclone` przez rpy2)

Sekcje powyżej trenują/walidują inny klasyfikator (RF/GNB na genach GO+CycleBase, Buettner et al.) na danych referencyjnych (`../../data/*.h5f`, których zresztą nie ma w tym repo) — to **nie** jest metoda użyta w `scHSC_herault` (`scHSC_herault/R_src/data_preparation.R::getCellCyclePhases`) i nie liczy niczego dla naszych komórek HSPC.

Ta sekcja odtwarza dokładnie krok z R: `scran::cyclone()` na surowych zliczeniach UMI, z wbudowanym w pakiet `mouse_cycle_markers.rds` (pary genów Scialdone et al.), wywołane przez `rpy2` na `merged_dataset.h5ad`. Sama implementacja bridge'a do R (`score_cell_cycle_with_scran_cyclone`) mieszka w [`src/utils/cell_cycle.py`](src/utils/cell_cycle.py) — nie w `src/utils/__init__.py`'s eagerowych importach, żeby import `src.utils` gdzie indziej (np. w `4. Trajectory Inference.ipynb`, który używa tylko `plot_streamlike`) nie zaczął nagle wymagać działającego rpy2+R. Wynik (`G1_score`/`S_score`/`G2M_score`/`phase` per komórka) to dokładnie to, czego szuka krok regresji cyklu komórkowego w `2. Data Integration.ipynb` (`CELL_CYCLE_SCORES_TSV`).

Różnice względem oryginału i dlaczego są nieszkodliwe:
- **Jeden przebieg zamiast czterech per próbka.** W R `getCellCyclePhases()` jest wołane osobno dla young_A/young_B/old_A/old_B (`Snakefile.py`, reguła `prepareCL` per `{smp}`). `cyclone()` liczy wynik z **rang par genów wewnątrz pojedynczej komórki** — nie zależy od innych komórek w tej samej macierzy — więc odpalenie go raz na całym `merged_dataset.h5ad` daje identyczne wyniki per komórka, tylko szybciej.
- **Mapowanie genów na Ensembl ID jest pozycyjne.** `mouse_cycle_markers.rds` jest kluczowany po Ensembl ID, a `merged_dataset.h5ad` ma w `var_names` symbole. Używamy `young_A_genes.tsv` (surowy `genes.tsv` z CellRangera) — zweryfikowałem, że jego kolejność wierszy odpowiada 1:1 kolejności genów w `merged_dataset.h5ad` (jedyna różnica to sufiksy `-1`/`-2` doklejone przez scanpy do 67 zduplikowanych symboli).
- **Wydajność.** `cyclone()` jednowątkowo to ~108 ms/komórkę (~27 min na ~14 900 komórek); z `BiocParallel::MulticoreParam` na tej maszynie ~23 ms/komórkę (~6 min). Wymaga `rpy2` + `bioconductor-scran` — dopisane do `environment.yml`.

Pełny przebieg na wszystkich 14 915 komórkach już wykonany: 13 580 G1_G0 / 949 G2_M / 386 S, 0 odrzuconych — `cell_cycle_scores.tsv` leży w repo (objęty `.gitignore`, tak jak inne wygenerowane pliki danych).

In [ ]:
import os
import sys
from pathlib import Path

# repo root na sys.path — konwencja z `4. Trajectory Inference.ipynb`
_ROOT = Path.cwd()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import scanpy as sc

from src.utils.cell_cycle import score_cell_cycle_with_scran_cyclone

MERGED_H5AD = Path("merged_dataset.h5ad")
GENES_TSV = Path("young_A_genes.tsv")  # genes.tsv z CellRangera, ten sam porządek genów co merged_dataset.h5ad
CELL_CYCLE_SCORES_TSV = Path("cell_cycle_scores.tsv")

N_CYCLONE_WORKERS = max((os.cpu_count() or 2) - 2, 1)

In [ ]:
adata = sc.read_h5ad(MERGED_H5AD)
assert np.issubdtype(adata.X.dtype, np.integer), "cyclone oczekuje surowych zliczeń UMI, nie znormalizowanych danych"

genes_tsv = pd.read_csv(GENES_TSV, sep="\t", header=None, names=["ensembl_id", "symbol"])
assert len(genes_tsv) == adata.n_vars, (
    f"{GENES_TSV} musi mieć tę samą liczbę wierszy co genów w {MERGED_H5AD} "
    "(mapowanie na Ensembl ID jest czysto pozycyjne, patrz sekcja wyżej)"
)
assert not genes_tsv["ensembl_id"].duplicated().any()

cc_scores = score_cell_cycle_with_scran_cyclone(
    counts=adata.X,
    gene_ids=genes_tsv["ensembl_id"].tolist(),
    cell_ids=adata.obs_names.tolist(),
    n_workers=N_CYCLONE_WORKERS,
)

n_before = len(cc_scores)
cc_scores = cc_scores.dropna()
print(
    f"{n_before - len(cc_scores)} / {n_before} komórek odrzuconych: za mało wykrytych genów, "
    "by cyclone przypisał fazę (odpowiednik filtra z R_src/prepareCL.R:82-93)"
)

cc_scores.to_csv(CELL_CYCLE_SCORES_TSV, sep="\t")
cc_scores["phase"].value_counts()